In [1]:
import os
import glob
import torch
from IPython.display import Audio, display
from source_separation import process_audio_file, process_folder_batch

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU available, using CPU")

CUDA available: True
Using GPU: NVIDIA GeForce RTX 3090
GPU memory: 25.21 GB


In [14]:
import os
import glob

def list_mp3_files(directory, limit=None):
    """List all .mp3 files (case-insensitive) in directory and its subdirectories."""
    # Get all files recursively
    paths = glob.glob(os.path.join(directory, "**", "*"), recursive=True)
    # Filter for files that are mp3s regardless of case
    mp3_files = [f for f in paths if os.path.isfile(f) and f.lower().endswith('.mp3')]
    if limit is not None and len(mp3_files) > limit:
        return mp3_files[:limit]
    return mp3_files

BASE_DIR = "/mnt/Aimir_HD/"
DATASETS = ["suno", "udio", "lastfm"]

print("Found the following collections:")
for dataset in DATASETS:
    audio_dir = os.path.join(BASE_DIR, dataset, "audio")
    mp3_files = list_mp3_files(audio_dir, limit=None)
    print(f"  - {dataset}: {len(mp3_files)} audio files")


Found the following collections:
  - suno: 187265 audio files
  - udio: 86007 audio files
  - lastfm: 68935 audio files


In [12]:
import os
import glob
import torch
from IPython.display import Audio, display
from source_separation import process_audio_file, process_folder_batch

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU available, using CPU")

def list_mp3_files(directory, limit=None):
    """List all .mp3 files (case-insensitive) in directory and its subdirectories."""
    # Get all files recursively
    paths = glob.glob(os.path.join(directory, "**", "*"), recursive=True)
    # Filter for files that are mp3s regardless of case
    mp3_files = [f for f in paths if os.path.isfile(f) and f.lower().endswith('.mp3')]
    if limit is not None and len(mp3_files) > limit:
        return mp3_files[:limit]
    return mp3_files

BASE_DIR = "/mnt/Aimir_HD/"
DATASETS = ["suno", "udio", "lastfm"]

# TEST MODE - Only process 5 files per collection
TEST_LIMIT = 5

print("TESTING MODE: Will process 5 files per collection")
print("Found the following collections:")
for dataset in DATASETS:
    audio_dir = os.path.join(BASE_DIR, dataset, "audio")
    mp3_files = list_mp3_files(audio_dir, limit=TEST_LIMIT)  # Apply limit here
    print(f" - {dataset}: Processing {len(mp3_files)} audio files for testing")

# Process limited files for each collection
for dataset in DATASETS:
    audio_dir = os.path.join(BASE_DIR, dataset, "audio")
    audio_files = list_mp3_files(audio_dir, limit=TEST_LIMIT)  # Apply limit here
    
    if len(audio_files) == 0:
        print(f"No audio files found in {audio_dir}")
        continue
    
    print(f"\nProcessing {len(audio_files)} test files from {dataset}")
    
    # Process each file individually to better track success/failure
    for i, file_path in enumerate(audio_files):
        file_id = os.path.basename(file_path).replace(".mp3", "")
        print(f"Processing file {i+1}/{len(audio_files)}: {file_id}")
        
        # Process the file
        success = process_audio_file(file_path, model_name="htdemucs")
        
        # Check results
        output_audio_dir = os.path.join(BASE_DIR, dataset, "segmented", file_id, "audio")
        if os.path.exists(output_audio_dir):
            print(f"Results saved to: {output_audio_dir}")
            # Verify the stems were created
            stems = glob.glob(os.path.join(output_audio_dir, "*.wav"))
            print(f"Found {len(stems)} stems: {[os.path.basename(s) for s in stems]}")
        else:
            print(f"ERROR: Output directory was not created at {output_audio_dir}")

print("\nTest processing complete. Please verify the output folders before proceeding with full dataset.")

CUDA available: True
Using GPU: NVIDIA GeForce RTX 3090
GPU memory: 25.21 GB
TESTING MODE: Will process 5 files per collection
Found the following collections:
 - suno: Processing 5 audio files for testing
 - udio: Processing 5 audio files for testing
 - lastfm: Processing 5 audio files for testing

Processing 5 test files from suno
Processing file 1/5: 6e41a72b-770e-4e88-9cee-cdab5481b312
Processing audio file: /mnt/Aimir_HD/suno/audio/6e41a72b-770e-4e88-9cee-cdab5481b312.mp3
Output directory: /mnt/Aimir_HD/suno/segmented/6e41a72b-770e-4e88-9cee-cdab5481b312/audio
Loading audio from /mnt/Aimir_HD/suno/audio/6e41a72b-770e-4e88-9cee-cdab5481b312.mp3...
Audio shape: torch.Size([2, 11520000]), Duration: 240.00s, Sample rate: 48000Hz
Loading pre-trained model 'htdemucs' on cuda...


Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to /root/.cache/torch/hub/checkpoints/955717e8-8726e21a.th
100%|██████████| 80.2M/80.2M [00:00<00:00, 107MB/s] 


Resampling from 48000Hz to 44100Hz...
Normalizing audio...
Normalized audio: peak from 0.5712 to 0.9000
Separating sources with segment size of 30s...
Processing audio in 10 segments with 20% overlap...
Processing segment 1/10 (0.0s - 30.0s)


/workspace/src/source_separation.py:98: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=device.type == 'cuda'):


Processing segment 2/10 (24.0s - 54.0s)


/workspace/src/source_separation.py:98: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast(enabled=device.type == 'cuda'):


Processing segment 3/10 (48.0s - 78.0s)
Processing segment 4/10 (72.0s - 102.0s)
Processing segment 5/10 (96.0s - 126.0s)
Processing segment 6/10 (120.0s - 150.0s)
Processing segment 7/10 (144.0s - 174.0s)
Processing segment 8/10 (168.0s - 198.0s)
Processing segment 9/10 (192.0s - 222.0s)
Processing segment 10/10 (216.0s - 240.0s)
Saving separated tracks to /mnt/Aimir_HD/suno/segmented/6e41a72b-770e-4e88-9cee-cdab5481b312/audio...
Normalized drums stem: peak 0.7948 -> 0.9
Saved drums to /mnt/Aimir_HD/suno/segmented/6e41a72b-770e-4e88-9cee-cdab5481b312/audio/drums.wav
Normalized bass stem: peak 0.5903 -> 0.9
Saved bass to /mnt/Aimir_HD/suno/segmented/6e41a72b-770e-4e88-9cee-cdab5481b312/audio/bass.wav
Normalized harmony stem: peak 0.4076 -> 0.9
Saved harmony to /mnt/Aimir_HD/suno/segmented/6e41a72b-770e-4e88-9cee-cdab5481b312/audio/harmony.wav
Normalized vocals stem: peak 0.6235 -> 0.9
Saved vocals to /mnt/Aimir_HD/suno/segmented/6e41a72b-770e-4e88-9cee-cdab5481b312/audio/vocals.wav
Sou

In [ ]:
# Base path and dataset definitions
BASE_DIR = "/workspace/samples"
DATASETS = ["suno_samples", "udio_samples", "lastfm_samples"]

# Function to list audio files
def list_audio_files(directory, limit=10):
    """List audio files in directory with optional limit"""
    files = glob.glob(os.path.join(directory, "*.mp3"))
    if limit and len(files) > limit:
        return files[:limit]
    return files

# Display info about all collections
print("Found the following collections:")
for dataset in DATASETS:
    audio_dir = os.path.join(BASE_DIR, dataset, "audio")
    audio_files = list_audio_files(audio_dir, limit=None)  # Count all files
    print(f"  - {dataset}: {len(audio_files)} audio files")

# Choose processing method
print("\nChoose processing method:")
print("1. Process a single file from each collection")
print("2. Process multiple files from each collection (batch)")

# Set your choice here (1 for single file, 2 for batch)
processing_mode = 2

# Process files for each collection
for dataset in DATASETS:
    audio_dir = os.path.join(BASE_DIR, dataset, "audio")
    audio_files = list_audio_files(audio_dir)
    
    if len(audio_files) == 0:
        print(f"No audio files found in {audio_dir}")
        continue
    
    if processing_mode == 1 and audio_files:
        # Process a single file from this collection
        file_index = 0
        selected_file = audio_files[file_index]
        file_id = os.path.basename(selected_file).replace(".mp3", "")
        
        print(f"\nProcessing file: {file_id} from {dataset}")
        
        # Process the file
        process_audio_file(selected_file, model_name="htdemucs")
        
        # Check results
        output_audio_dir = os.path.join(BASE_DIR, dataset, "segmented", file_id, "audio")
        if os.path.exists(output_audio_dir):
            print(f"Results saved to: {output_audio_dir}")
    
    elif processing_mode == 2:
        # Process all files in this collection in batch
        # print(f"\nProcessing all files in {dataset} with batch mode")
        
        # Process the files batch
        process_folder_batch(dataset, base_path=BASE_DIR, model_name="htdemucs")

print("\nProcessing complete for all collections!")